In [1]:
!pip install torch torchaudio transformers

In [2]:
import torch
import torchaudio
from transformers import AutoModel, Wav2Vec2FeatureExtractor

In [3]:
!curl -L -C - "https://zenodo.org/records/6771120/files/tinyAAM.zip?download=1" -o sample_data/tinyAAM.zip

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  160M  100  160M    0     0   628k      0  0:04:21  0:04:21 --:--:--  588k


In [4]:
import zipfile

with zipfile.ZipFile("sample_data/tinyAAM.zip") as z:
    z.extractall("sample_data/tinyAAM")

In [5]:
from pathlib import Path

FILES_PATH = Path("sample_data/tinyAAM/audio-mixes-mp3")

files = sorted(f for f in FILES_PATH.rglob("*"))

In [6]:
import os
from google.colab import userdata
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [7]:
!pip install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 26.0 MB/s eta 0:00:00


In [8]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

In [9]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

In [10]:
MODEL_ID = "m-a-p/MERT-v1-330M"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)

if torch.cuda.is_available():
  model = model.to(device).eval()

preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.03k [00:00<?, ?B/s]

configuration_MERT.py:   0%|          | 0.00/5.34k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-330M:
- configuration_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_MERT.py:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-330M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.26GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/403 [00:00<?, ?it/s]

In [13]:
def chunk_audio_file(wav, audio_sample_rate):
  if audio_sample_rate != SAMPLE_RATE_MERT:
    wav = torchaudio.functional.resample(wav, audio_sample_rate, SAMPLE_RATE_MERT)

  duration = wav.numel() / SAMPLE_RATE_MERT
  chunk = int(CHUNK_SECONDS * SAMPLE_RATE_MERT)
  hop = max(1, int((CHUNK_SECONDS - OVERLAP_SECONDS) * SAMPLE_RATE_MERT))

  # Add padding on short audios
  if wav.numel() < chunk:
    wav = torch.nn.functional.pad(wav, (0, chunk - wav.numel()))

  # Chunk audio parts
  return [wav[i : i + chunk] for i in range(0, wav.numel() - chunk + 1, hop)], duration

In [30]:
def audio_embeddings(chunks):
  per_chunk = []

  for j in range(0, len(chunks), BATCH_SIZE):
      batch = [c.numpy() for c in chunks[j : j + BATCH_SIZE]]
      inputs = processor(
          batch,
          sampling_rate=SAMPLE_RATE_MERT,
          return_tensors="pt",
          padding=True,
      )
      inputs = {k: v.to(device) for k, v in inputs.items()}

      # inference_mode: no autograd graph, no retained activations
      # autocast: fp16 conv + attention, roughly halves activation memory
      with torch.inference_mode():
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
          out = model(**inputs, output_hidden_states=True)

          # index the tuple directly -- do NOT torch.stack all 25 layers
          hidden = out.hidden_states[12]   # (batch, frames, 1024)
          vec = hidden.mean(dim=1)                   # (batch, 1024)

      per_chunk.append(vec.float().cpu())
      del out, hidden, vec, inputs

  emb = torch.cat(per_chunk, dim=0).mean(dim=0)
  return torch.nn.functional.normalize(emb, dim=0)

In [31]:
vectors = []
SAMPLE_RATE_MERT = 24000
CHUNK_SECONDS = 10.0
OVERLAP_SECONDS = 0.0
BATCH_SIZE = 1

for i, cfile in enumerate(files):
  wav, sr = torchaudio.load(str(cfile))
  wav = wav.mean(dim=0)

  chunks, duration = chunk_audio_file(wav, audio_sample_rate=sr)
  embeddings = audio_embeddings(chunks)

  vectors.append({
    "id": f"track-{i}",
    "values": embeddings.tolist(),
    "metadata": {
        "filename": cfile.name,
        "path": cfile.relative_to(FILES_PATH).as_posix(),
        "duration_seconds": round(duration, 2)
    },
  })

In [32]:
PINECONE_INDEX_NAME = "music-recomendations"
PINECONE_NAMESPACE = "tiny-audios"

# Create new pinecone index
if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=1024, # Standard dimensions for MERT embeddings
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(PINECONE_INDEX_NAME)

In [33]:
VECTOR_SIZE = 100

for i in range(0, len(vectors), VECTOR_SIZE):
    index.upsert(vectors=vectors[i : i + VECTOR_SIZE], namespace=PINECONE_NAMESPACE)

In [35]:
def query_track_similaries(file_path):
  wav, sr = torchaudio.load(str(file_path))
  wav = wav.mean(dim=0)
  track_chunks, duration = chunk_audio_file(wav, audio_sample_rate=sr)
  track_emb = audio_embeddings(track_chunks)

  return index.query(
      vector=track_emb.tolist(),
      top_k=3,
      namespace=PINECONE_NAMESPACE,
      include_metadata=True,
  )

print(query_track_similaries(Path("sample_data/tinyAAM/audio-mixes-mp3/0758_mix.mp3")))

QueryResponse(matches=[ScoredVector(id='track-4', score=1.00096607, values=[], metadata={'duration_seconds': 147.93, 'filename': '0758_mix.mp3', 'path': '0758_mix.mp3'}), ScoredVector(id='track-7', score=0.989798546, values=[], metadata={'duration_seconds': 162.72, 'filename': '1050_mix.mp3', 'path': '1050_mix.mp3'}), ScoredVector(id='track-17', score=0.972939074, values=[], metadata={'duration_seconds': 176.72, 'filename': '2841_mix.mp3', 'path': '2841_mix.mp3'})], namespace='tiny-audios', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 04 Aug 2026 14:15:13 GMT', 'content-type': 'application/json', 'content-length': '484', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '2', 'x-pinecone-request-latency-ms': '265', 'x-envoy-upstream-service-time': '49', 'x-pinecone-response-duration-ms': '267', 'grpc-status': '0', 'server': 'envoy'}))
